# Degenerate Perturbation Theory: the Hydrogen Stark Effect

**Phase 4** (final phase) of `Perturbation_and_Basis_Methods_Plan.md`. The `n=2`
hydrogen shell is 4-fold degenerate: `2s`, `2p_x`, `2p_y`, `2p_z`
(`hydrogen.py`), all at `E_2=-1/8`. A uniform external field along `z` adds
`H'=E_field*z` -- since this couples degenerate states, ordinary (non-degenerate)
perturbation theory's `1/(E_n-E_m)` energy denominators would divide by zero.
The fix (degenerate perturbation theory) is to diagonalize `H'` *within* the
degenerate subspace first: whichever basis of the subspace diagonalizes `H'`
there gives the correct zeroth-order states, and its eigenvalues are the
1st-order energy shifts.

This replaces the purely cosmetic legacy `Hydrogen Atom.ipynb` (which only
plotted a spherical harmonic's shape) with genuine physics, and everything is
computed directly from `hydrogen.py`'s actual wavefunctions -- no textbook
constant is hardcoded; the well-known `+/-3` (atomic units) result is a
*prediction to check against*, not an input.

In [1]:
import sys
sys.path.insert(0, r".")
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

import hydrogen as hyd
import perturbation as pt

MEDIA_DIR = Path('media')
MEDIA_DIR.mkdir(exist_ok=True)


## Building the 4x4 `H'=z` matrix

Quadrature: Simpson in `r`, Gauss-Legendre in `cos(theta)` (exact for these
smooth angular functions, as in `Validation.ipynb`), uniform in `phi`. The
integrand needs the full spherical volume element `r^2*sin(theta)*dr*dtheta*
dphi` -- the Gauss-Legendre substitution `u=cos(theta)` already absorbs
`sin(theta)*dtheta` into its weights, but the `r^2` from the radial part of
the volume element still has to be included explicitly alongside the plain
`dr` Simpson integration.

In [2]:
r = np.linspace(1e-6, 40.0, 800)
deg = 24
u, w_u = np.polynomial.legendre.leggauss(deg)
theta_nodes = np.arccos(u)
n_phi = 32
phi_nodes = np.linspace(0, 2 * np.pi, n_phi, endpoint=False)
dphi = 2 * np.pi / n_phi

R, TH, PH = np.meshgrid(r, theta_nodes, phi_nodes, indexing='ij')

def overlap(f, g):
    integrand = f * g * R ** 2  # r^2 volume-element factor
    step1 = dphi * np.sum(integrand, axis=2)      # integrate phi
    step2 = np.sum(w_u[None, :] * step1, axis=1)   # integrate theta (Gauss-Legendre)
    return simpson(step2, x=r)                      # integrate r

from scipy.integrate import simpson

psi_2s = hyd.wavefunction(2, 0, 0, R, TH, PH).real
p_x, p_y, p_z = hyd.real_p_orbitals(2, R, TH, PH)
states = {'2s': psi_2s, '2p_x': p_x.real, '2p_y': p_y.real, '2p_z': p_z.real}
names = ['2s', '2p_x', '2p_y', '2p_z']
z_op = R * np.cos(TH)


In [3]:
results = []
def check(name, cond, detail=""):
    results.append((name, bool(cond), detail))
    print(f"{'PASS' if cond else 'FAIL'}: {name}  {detail}")


In [4]:
# --- Check A: each n=2 state is correctly normalized on this grid ---
norms = {name: overlap(states[name], states[name]) for name in names}
for name, n in norms.items():
    print(f"  <{name}|{name}> = {n:.6f}")
max_norm_err = max(abs(n - 1.0) for n in norms.values())
check("A n=2 states normalized", max_norm_err < 1e-4, f"max|norm-1|={max_norm_err:.2e}")


  <2s|2s> = 1.000000
  <2p_x|2p_x> = 1.000000
  <2p_y|2p_y> = 1.000000
  <2p_z|2p_z> = 1.000000
PASS: A n=2 states normalized  max|norm-1|=2.09e-07


In [5]:
# --- Check B: H' = z's matrix in this basis has the expected selection-
# rule sparsity -- only <2s|z|2p_z> (and its symmetric partner) nonzero,
# from Delta(m)=0 and the parity of z -- computed here, not assumed ---
H_prime = np.zeros((4, 4))
for i, ni in enumerate(names):
    for j, nj in enumerate(names):
        H_prime[i, j] = overlap(states[ni], z_op * states[nj])
print(np.round(H_prime, 5))

off_pattern = H_prime.copy()
off_pattern[0, 3] = off_pattern[3, 0] = 0.0  # zero out the one expected-nonzero pair
max_other = np.max(np.abs(off_pattern))
check("B only <2s|z|2p_z> is nonzero (selection rules)", max_other < 1e-3, f"max other element={max_other:.2e}")


[[-0.  0.  0. -3.]
 [ 0. -0. -0. -0.]
 [ 0.  0. -0.  0.]
 [-3. -0.  0. -0.]]
PASS: B only <2s|z|2p_z> is nonzero (selection rules)  max other element=1.63e-16


## Diagonalizing within the degenerate subspace

`perturbation.degenerate_perturbation` diagonalizes `H'` directly (it's already
the full 4x4 matrix for this degenerate subspace) -- eigenvalues are the
1st-order energy shifts per unit field, eigenvectors are the correct
zeroth-order states.

In [6]:
shifts, eigvecs = pt.degenerate_perturbation(H_prime)
print("shifts (per unit E_field):", np.round(shifts, 5))
print("eigenvectors (columns, basis order 2s,2p_x,2p_y,2p_z):")
print(np.round(eigvecs, 4))

check("C two states shift by +/-3 (atomic units)",
      np.allclose(np.sort(np.abs(shifts))[-2:], [3.0, 3.0], atol=1e-3),
      f"shifts={np.round(shifts, 4)}")
check("C two states are unshifted (2p_x, 2p_y)",
      np.allclose(np.sort(np.abs(shifts))[:2], [0.0, 0.0], atol=1e-3),
      f"shifts={np.round(shifts, 4)}")


shifts (per unit E_field): [-3. -0. -0.  3.]
eigenvectors (columns, basis order 2s,2p_x,2p_y,2p_z):
[[-0.7071 -0.      0.      0.7071]
 [ 0.      0.9996  0.0269  0.    ]
 [ 0.     -0.0269  0.9996 -0.    ]
 [-0.7071  0.      0.     -0.7071]]
PASS: C two states shift by +/-3 (atomic units)  shifts=[-3. -0. -0.  3.]
PASS: C two states are unshifted (2p_x, 2p_y)  shifts=[-3. -0. -0.  3.]


In [7]:
# --- Check D: the shifted eigenvectors are (2s +/- 2p_z)/sqrt(2) ---
shifted_idx = np.argsort(np.abs(shifts))[-2:]
expected = 1 / np.sqrt(2)
overlaps = []
for idx in shifted_idx:
    v = eigvecs[:, idx]
    # component structure should be (+-1/sqrt2, 0, 0, +-1/sqrt2)
    weight_2s_2pz = abs(v[0]) + abs(v[3])
    weight_2px_2py = abs(v[1]) + abs(v[2])
    overlaps.append((weight_2s_2pz, weight_2px_2py))
    print(f"eigenvector {idx}: {np.round(v, 4)}  |2s|+|2pz|={weight_2s_2pz:.4f}  |2px|+|2py|={weight_2px_2py:.4f}")

check("D shifted eigenvectors are pure (2s,2p_z) combinations",
      all(abs(w2 - 2 * expected) < 1e-3 and w1 < 1e-3 for w2, w1 in overlaps),
      f"{overlaps}")


eigenvector 0: [-0.7071  0.      0.     -0.7071]  |2s|+|2pz|=1.4142  |2px|+|2py|=0.0000
eigenvector 3: [ 0.7071  0.     -0.     -0.7071]  |2s|+|2pz|=1.4142  |2px|+|2py|=0.0000
PASS: D shifted eigenvectors are pure (2s,2p_z) combinations  [(np.float64(1.414213562373095), np.float64(3.4920293661857055e-18)), (np.float64(1.414213562373095), np.float64(2.774849392473426e-18))]


In [8]:
n_pass = sum(1 for _, ok, _ in results if ok)
print(f"\n{n_pass}/{len(results)} Phase 4 checks passed")
assert n_pass == len(results), "Phase 4 validation failed"



5/5 Phase 4 checks passed


## Stark diagram

Since the degenerate-subspace eigenvalues of `z` are fixed, the actual energy
shift for a field of strength `E_field` is just `E_field * shift` -- exact
linear (first-order) Stark splitting, no further computation needed to sweep
the field.

In [9]:
E_field = np.linspace(-0.1, 0.1, 100)
fig, ax = plt.subplots(figsize=(7, 5))
for shift in np.unique(np.round(shifts, 6)):
    label = f'shift = {shift:+.3f}' + ('  (2p_x, 2p_y)' if abs(shift) < 1e-6 else '  ((2s+2p_z)/sqrt2 or (2s-2p_z)/sqrt2)')
    ax.plot(E_field, E_field * shift, label=label)
ax.set_xlabel('E_field (atomic units)')
ax.set_ylabel('Energy shift from E_2')
ax.set_title('Linear Stark effect, hydrogen n=2')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(MEDIA_DIR / 'stark_diagram.png', dpi=150)
plt.show()


C:\Users\Hasan's Laptop\AppData\Local\Temp\ipykernel_29224\1767014222.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
